# Xem dữ liệu thời tiết và thành phố từ Database
Đoạn code dưới đây sẽ kết nối thẳng vào database PostgreSQL đang chạy bằng Docker trên máy (qua cổng `localhost:5432`) để trích xuất và hiển thị dữ liệu bằng `pandas`.

In [167]:
import pandas as pd
import psycopg2

# Kết nối vào Postgres server đang được Docker expose ra host qua cổng 5432
conn = psycopg2.connect(
    host="localhost",
    port="5432",
    user="postgres",
    password="Intelligent_Data_Analysis",
    dbname="weather"
)

# Lấy 5 dòng đầu từ bảng cities
print("--- Dữ liệu từ bảng Cities ---")
cities_df = pd.read_sql_query("SELECT * FROM public.cities LIMIT 5;", conn)
display(cities_df)

# số thành phố đã lưu trong bảng weather_daily
total_cities = pd.read_sql_query("SELECT COUNT(DISTINCT city_id) as total_cities FROM public.weather_daily;", conn)
print(f"\nTổng số thành phố có dữ liệu thời tiết đang lưu: {total_cities['total_cities'][0]}")
# Lấy 5 dòng dữ liệu mới nhất
count_city_df = pd.read_sql_query("SELECT city_id, COUNT(*) as record_count FROM public.weather_daily GROUP BY city_id ", conn)
display(count_city_df)


# Thống kê tổng số dòng dữ liệu
total_rows = pd.read_sql_query("SELECT COUNT(*) as total_records FROM public.weather_daily ORDER BY COUNT(*) DESC;", conn)
print(f"\nTổng số bản ghi thời tiết đang lưu: {total_rows['total_records'][0]}")

# Đóng kết nối DB
conn.close()

--- Dữ liệu từ bảng Cities ---


C:\Users\trung\AppData\Local\Temp\ipykernel_37764\1820250221.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cities_df = pd.read_sql_query("SELECT * FROM public.cities LIMIT 5;", conn)


,city_id,city,country,latitude,longitude
0,168,An Giang,Vietnam,10.5185,105.1152
1,327,Bà Rịa – Vũng Tàu,Vietnam,10.4114,107.1362
2,849,Bắc Giang,Vietnam,21.2731,106.1946
3,193,Bắc Kạn,Vietnam,22.1470,105.8326
4,496,Bạc Liêu,Vietnam,9.2941,105.7242



Tổng số thành phố có dữ liệu thời tiết đang lưu: 63


C:\Users\trung\AppData\Local\Temp\ipykernel_37764\1820250221.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  total_cities = pd.read_sql_query("SELECT COUNT(DISTINCT city_id) as total_cities FROM public.weather_daily;", conn)
C:\Users\trung\AppData\Local\Temp\ipykernel_37764\1820250221.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  count_city_df = pd.read_sql_query("SELECT city_id, COUNT(*) as record_count FROM public.weather_daily GROUP BY city_id ", conn)


,city_id,record_count
0,347,2291
1,620,2291
2,684,2291
3,645,2291
4,951,2291
...,...,...
58,456,2291
59,255,2291
60,811,2291
61,813,2291



Tổng số bản ghi thời tiết đang lưu: 144333


C:\Users\trung\AppData\Local\Temp\ipykernel_37764\1820250221.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  total_rows = pd.read_sql_query("SELECT COUNT(*) as total_records FROM public.weather_daily ORDER BY COUNT(*) DESC;", conn)


In [ ]:
# import psycopg2

# # Kết nối vào Postgres server
# conn = psycopg2.connect(
#     host="localhost",
#     port="5432",
#     user="postgres",
#     password="Intelligent_Data_Analysis",
#     dbname="weather"
# )

# # Bật autocommit để thực thi các lệnh DROP/TRUNCATE mà không cần commit thủ công
# conn.autocommit = True
# cursor = conn.cursor()

# try:
#     # Bỏ "IF EXISTS" vì TRUNCATE trong Postgres không hỗ trợ từ khóa này
#     cursor.execute("TRUNCATE TABLE public.weather_daily CASCADE;")
#     cursor.execute("TRUNCATE TABLE public.cities CASCADE;")
#     print("Đã xóa sạch dữ liệu trong các bảng thành công!")
# except Exception as e:
#     print(f"Có lỗi xảy ra: {e}")

Đã xóa sạch dữ liệu trong các bảng thành công!


In [2]:
import psycopg2

# Danh sách các ID cần xóa (Ví dụ: [101, 102, 103])
ids_to_delete = [168]

# Kết nối vào Postgres server
conn = psycopg2.connect(
    host="aws-1-ap-northeast-2.pooler.supabase.com",
    port="6543",
    user="postgres.lzkbkkhhuyvefjqemdtk",
    password="Trunghieu06012005",
    dbname="postgres"
)

# Bật autocommit để thực thi lệnh ngay lập tức
conn.autocommit = True
cursor = conn.cursor()

try:
    # Xóa hàng trong bảng weather_daily dựa trên danh sách ID
    # Sử dụng = ANY(%s) là cách an toàn và hiệu quả nhất để truyền list Python vào SQL
    cursor.execute("DELETE FROM public.weather_daily WHERE city_id = ANY(%s);", (ids_to_delete,))
    
    # Xóa hàng trong bảng cities dựa trên danh sách ID
    cursor.execute("DELETE FROM public.cities WHERE city_id = ANY(%s);", (ids_to_delete,))
    
    print(f"Đã xóa thành công các hàng có ID thuộc danh sách: {ids_to_delete}")

except Exception as e:
    print(f"Có lỗi xảy ra trong quá trình xóa: {e}")
finally:
    cursor.close()
    conn.close()

Đã xóa thành công các hàng có ID thuộc danh sách: [168]
